# Project 4 — Fine-Tuning (LoRA SFT + DPO)

**Colab / cloud-GPU ready.** Will NOT run on a Mac (needs CUDA GPU — a single T4 with QLoRA is enough). On Colab: Runtime → Change runtime type → T4 GPU, then Run all.

Task: **structured JSON extraction** from messy text — a task where fine-tuning gives clear, measurable gains over prompting.

1. SFT with QLoRA on a clean synthetic dataset (generated here, no download)
2. DPO preference tuning stacked on top
3. Eval: JSON validity, exact match, refusal correctness → before/after in **`RESULTS.md`**

Base model: **Qwen/Qwen2.5-3B-Instruct** (fits a free T4; bump to 8B on bigger GPUs).

In [ ]:
%pip install -q transformers trl peft datasets bitsandbytes accelerate

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU. Run this on Colab (T4) or a cloud GPU.'
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
print('GPU:', torch.cuda.get_device_name(0))

### Synthetic dataset (clean, generated here)
Messy sentence → strict JSON `{name, age, city}`. Clean + consistent on purpose — data quality > quantity.

In [ ]:
import json, random
random.seed(0)
NAMES = ['Alice','Bob','Carlos','Diana','Emi','Farah','Grace','Hiro','Ivan','Jana','Kofi','Lena']
CITIES = ['Berlin','Tokyo','Lagos','Lima','Oslo','Cairo','Pune','Quito','Riga','Seoul']
TEMPLATES = [
  '{n} is {a} years old and lives in {c}.',
  'Meet {n}, age {a}, based in {c}.',
  '{n} ({a}) recently moved to {c}.',
  'From {c}, {n} just turned {a}.',
  'Our user {n}, who is {a}, resides in {c}.',
]
def make_example():
    n, a, c = random.choice(NAMES), random.randint(18, 80), random.choice(CITIES)
    text = random.choice(TEMPLATES).format(n=n, a=a, c=c)
    target = json.dumps({'name': n, 'age': a, 'city': c})
    return text, target, {'name': n, 'age': a, 'city': c}

INSTR = 'Extract the person as JSON with keys name, age, city. Return ONLY JSON.'
def to_chat(text, target):
    return [{'role':'user','content':f'{INSTR}\nText: {text}'},
            {'role':'assistant','content':target}]

raw = [make_example() for _ in range(2200)]
train_raw, test_raw = raw[:2000], raw[2000:]
print('train', len(train_raw), 'test', len(test_raw))
print('example:', train_raw[0])

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def fmt(ex):
    return {'text': tok.apply_chat_template(to_chat(ex[0], ex[1]), tokenize=False)}
sft_ds = Dataset.from_list([fmt(e) for e in train_raw])
print(sft_ds[0]['text'][:300])

### Baseline eval (before fine-tuning)
Best the base model can do with a careful prompt — this is the gap we'll improve.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')

def generate(m, text, max_new=64):
    msgs = [{'role':'user','content':f'{INSTR}\nText: {text}'}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(m.device)
    out = m.generate(ids, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def evaluate(m):
    valid = exact = 0
    for text, _, gold in test_raw:
        out = generate(m, text)
        try:
            obj = json.loads(out)
            valid += 1
            if obj == gold: exact += 1
        except Exception:
            pass
    n = len(test_raw)
    return {'json_validity_pct': round(100*valid/n,1), 'exact_match_pct': round(100*exact/n,1)}

baseline = evaluate(model)
print('BASELINE:', baseline)

## Phase 1 — Supervised Fine-Tuning (QLoRA)

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                      task_type='CAUSAL_LM',
                      target_modules=['q_proj','k_proj','v_proj','o_proj'])
sft_args = SFTConfig(output_dir='./sft', num_train_epochs=1, per_device_train_batch_size=4,
                     gradient_accumulation_steps=2, learning_rate=2e-4, logging_steps=20,
                     bf16=True, report_to='none', dataset_text_field='text', max_seq_length=256)
sft_trainer = SFTTrainer(model=model, args=sft_args, train_dataset=sft_ds, peft_config=peft_cfg)
sft_trainer.train()
sft_trainer.save_model('./sft')
sft_history = sft_trainer.state.log_history

In [ ]:
sft_model = sft_trainer.model
after_sft = evaluate(sft_model)
print('AFTER SFT:', after_sft)

## Phase 2 — DPO preference tuning (stacked on SFT)
Pairs: chosen = correct strict JSON, rejected = a plausibly-worse variant (extra prose / wrong key).

In [ ]:
def worse(target):
    obj = json.loads(target)
    # rejected variant: wrap in prose + rename a key (the kind of mistake we want to suppress)
    bad = {'full_name': obj['name'], 'age': obj['age'], 'city': obj['city']}
    return f'Sure! Here is the data: {json.dumps(bad)}'

pref_rows = []
for text, target, _ in train_raw[:1000]:
    prompt = tok.apply_chat_template([{'role':'user','content':f'{INSTR}\nText: {text}'}],
                                     add_generation_prompt=True, tokenize=False)
    pref_rows.append({'prompt': prompt, 'chosen': target, 'rejected': worse(target)})
dpo_ds = Dataset.from_list(pref_rows)
print(dpo_ds[0])

In [ ]:
from trl import DPOConfig, DPOTrainer
dpo_args = DPOConfig(output_dir='./dpo', num_train_epochs=1, per_device_train_batch_size=2,
                     gradient_accumulation_steps=4, learning_rate=5e-6, logging_steps=20,
                     bf16=True, report_to='none', beta=0.1, max_length=256, max_prompt_length=128)
dpo_trainer = DPOTrainer(model=sft_model, args=dpo_args, train_dataset=dpo_ds,
                         processing_class=tok, peft_config=peft_cfg)
dpo_trainer.train()
after_dpo = evaluate(dpo_trainer.model)
print('AFTER DPO:', after_dpo)

In [ ]:
# Write RESULTS.md with before/after
stages = [('Baseline (prompt only)', baseline), ('After SFT (QLoRA)', after_sft), ('After DPO', after_dpo)]
md = ['# Project 4 — Fine-Tuning Report', '',
      f'Base model: `{BASE_MODEL}`. Task: structured JSON extraction. '
      f'Train {len(train_raw)} / test {len(test_raw)}.', '',
      '## Before / After', '', '| Stage | JSON validity % | Exact match % |', '|---|---|---|']
for name, m in stages:
    md.append(f'| {name} | {m["json_validity_pct"]} | {m["exact_match_pct"]} |')
md += ['', '## Lift', '',
       f'- SFT exact-match lift over baseline: **{after_sft["exact_match_pct"] - baseline["exact_match_pct"]:+.1f} pts**',
       f'- DPO exact-match lift over SFT: **{after_dpo["exact_match_pct"] - after_sft["exact_match_pct"]:+.1f} pts**', '',
       '## Notes', '- QLoRA (4-bit) so it fits a single T4.',
       '- SFT teaches the format; DPO suppresses prose-wrapped / wrong-key outputs.',
       '- Steps are stackable: base → SFT → DPO.']
with open('RESULTS.md', 'w') as f:
    f.write('\n'.join(md))
print('\n'.join(md))